# Evaluator-Optimizer Loop: Evaluation Notebook
**Sprints Multi-Agent AI Career & Assessment Orchestrator**

**Analogy:** a teacher (generator) writes feedback, an editor (LLM judge) scores it against a checklist, and the teacher rewrites it.
To control cost, the editor's patience ends after **3 rewrites** (acceptance criterion 5).

This notebook proves four acceptance criteria with evidence:
1. Loop halts after at most 3 refinements (even with a judge that never approves).
2. The loop actually improves feedback quality when the judge is reasonable.
3. ChromaDB retrieval stays under 1000 ms (acceptance criterion 3).
4. Tool failures return a structured `Unable to verify` (acceptance criterion 4).

> It runs with **Gemini** when `GOOGLE_API_KEY` is set in `.env`; otherwise it uses the deterministic offline stand-in (the saved outputs were produced offline).

In [1]:
import os, sys, time, statistics, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)                                 # make `import src` work
from src.llm import is_offline
print("Mode:", "OFFLINE stand-in" if is_offline() else "LIVE Gemini:" + os.getenv("GEMINI_MODEL", ""))

Mode: OFFLINE stand-in


## 1. A flawed learner submission and the live sandbox result

In [2]:
from src.tools.code_repl import run_python_tests
BAD = '''def average(numbers):
    return sum(numbers) / (len(numbers) + 1)   # bug: divides by n+1
'''
TESTS = "assert average([2, 4, 6]) == 4\nassert average([10]) == 10\nassert average([1, 2, 3, 4]) == 2.5"
print(run_python_tests(BAD, TESTS).model_dump())

{'passed': 0, 'total': 3, 'stdout': '', 'errors': ['assert average([2, 4, 6]) == 4 -> AssertionError: ', 'assert average([10]) == 10 -> AssertionError: ', 'assert average([1, 2, 3, 4]) == 2.5 -> AssertionError: '], 'timed_out': False}


## 2. Run the full evaluator-optimizer loop and show every round

In [3]:
from src.agents import assessor_agent
from src.agents.assessor_agent import build_assessor_graph
final = None
# stream_mode='updates' yields one dict per node execution, so we can print the loop step by step
for step in build_assessor_graph().stream({"code": BAD, "tests": TESTS, "learner_id": "SPR-1004"}, stream_mode="updates"):
    node, upd = next(iter(step.items()))
    if node == "evaluate":
        j = upd["judgement"]; print(f"[evaluate] judge score={j['score']}/10 approved={j['approved']} critique={j['critique']}")
    elif node == "refine":
        print(f"[refine  ] refinement #{upd['refinements']}: {upd['draft'][:110]}...")
    elif node == "draft":
        print(f"[draft   ] {upd['draft'][:110]}")
    elif node == "finalize":
        final = upd["report"]
print("\nFINAL:", json.dumps({k: final[k] for k in ("score", "judge_score", "refinements_used", "halted_by_cap")}))

[draft   ] Tests passed: 0/3
[evaluate] judge score=5/10 approved=False critique=gives a concrete next step; links to curriculum
[refine  ] refinement #1: Tests passed: 0/3 Next step: fix the failing case and re-run the tests. Review Module 1 (Python Foundations) a...
[evaluate] judge score=10/10 approved=True critique=meets the bar

FINAL: {"score": 10, "judge_score": 10, "refinements_used": 1, "halted_by_cap": false}


## 3. Proof of the hard cap: a judge that NEVER approves
We swap in a stub judge. The loop must stop after exactly 3 refinements (4 judgements) and flag `halted_by_cap`.

In [4]:
from src.state import Judgement, MAX_REFINEMENTS
class StubbornLLM:
    judge_calls = 0; refine_calls = 0
    def draft_feedback(self, facts): return "draft"
    def judge_feedback(self, facts, draft):
        StubbornLLM.judge_calls += 1
        return Judgement(score=2, approved=False, critique="never good enough")
    def refine_feedback(self, facts, draft, critique):
        StubbornLLM.refine_calls += 1
        return draft + "+"
original = assessor_agent.get_llm
assessor_agent.get_llm = lambda: StubbornLLM()
rep = assessor_agent.run_assessor(BAD, TESTS)
assessor_agent.get_llm = original
print(f"MAX_REFINEMENTS={MAX_REFINEMENTS} | refinements_used={rep.refinements_used} | judge_calls={StubbornLLM.judge_calls} | refine_calls={StubbornLLM.refine_calls} | halted_by_cap={rep.halted_by_cap}")
assert rep.refinements_used == 3 and StubbornLLM.refine_calls == 3 and rep.halted_by_cap
print("PASS: loop halted strictly at 3 refinements")

MAX_REFINEMENTS=3 | refinements_used=3 | judge_calls=4 | refine_calls=3 | halted_by_cap=True
PASS: loop halted strictly at 3 refinements


## 4. Rubric scoring example (correctness 50 + docstrings 20 + error handling 20 + feedback clarity 10)

In [5]:
GOOD = '''def average(numbers):
    \"\"\"Return the arithmetic mean of a non-empty list.\"\"\"
    if not numbers:
        raise ValueError("numbers must not be empty")
    return sum(numbers) / len(numbers)
'''
for label, code in [("buggy", BAD), ("good", GOOD)]:
    r = assessor_agent.run_assessor(code, TESTS)
    print(f"{label:6} score={r.score:3}/100 tests={r.tests.passed}/{r.tests.total} refinements={r.refinements_used}")

buggy  score= 10/100 tests=0/3 refinements=1


good   score=100/100 tests=3/3 refinements=1


## 5. ChromaDB retrieval latency (budget: 1000 ms)

In [6]:
from src.tools.rag_retriever import get_retriever
r = get_retriever()
qs = ["passing score python assignment", "how many refinement iterations", "chunk size and overlap for RAG", "who approves a placement",
      "why are SQL tools read only", "MemorySaver thread_id isolation", "employer partners", "ReAct loop", "top-k retrieval", "interrupt and resume"] * 5
times = [r.retrieve(q).latency_ms for q in qs]
q = statistics.quantiles(times, n=20)
print(f"queries={len(times)} chunks_in_index={r.collection.count()} embedder={r.embedder.name}")
print(f"median={statistics.median(times):.2f} ms | p95={q[-1]:.2f} ms | max={max(times):.2f} ms | budget=1000 ms")
assert max(times) < 1000; print("PASS: all retrievals under 1000 ms")

queries=50 chunks_in_index=16 embedder=hash1024
median=0.49 ms | p95=1.08 ms | max=1.71 ms | budget=1000 ms
PASS: all retrievals under 1000 ms


## 6. Graceful failure: SQLite and the Python REPL

In [7]:
from pathlib import Path
from src.tools import db_queries
import subprocess
print("SQLite down  ->", db_queries.fetch_learner("SPR-1001", db_path=Path("/nonexistent/x.db")).model_dump())
real_run = subprocess.run
subprocess.run = lambda *a, **k: (_ for _ in ()).throw(OSError("sandbox unavailable"))
print("REPL down    ->", run_python_tests(GOOD, TESTS).model_dump())
subprocess.run = real_run

SQLite down  -> {'status': 'unable_to_verify', 'message': 'Unable to verify', 'tool': 'sqlite', 'reason': 'OperationalError: unable to open database file'}
REPL down    -> {'status': 'unable_to_verify', 'message': 'Unable to verify', 'tool': 'python_repl', 'reason': 'OSError: sandbox unavailable'}


## 7. HITL escalation through the orchestrator

In [8]:
from langgraph.types import Command
from src.agents.orchestrator import build_orchestrator
app = build_orchestrator(); cfg = {"configurable": {"thread_id": "nb-demo"}}
out = app.invoke({"user_input": "Please approve placement for SPR-1001"}, cfg)
print("Paused for human:", "__interrupt__" in out)
print(json.dumps(out["__interrupt__"][0].value["recommendation"], indent=2))
out = app.invoke(Command(resume={"decision": "approve", "reviewer": "Nour (Career Advisor)", "notes": "Confirmed."}), cfg)
print(out["final_response"])

Paused for human: True
{
  "learner_id": "SPR-1001",
  "name": "Ahmed Youssef",
  "readiness_status": "Ready for Placement",
  "python_score": 95,
  "completed_modules": 12,
  "target_salary_egp": 30000,
  "recommended_employers": [
    "Fawry",
    "Swvl"
  ],
  "ai_rationale": "Ahmed Youssef completed 12/15 modules with a Python score of 95, meeting the Ready for Placement thresholds (>=12 modules, >=80).",
  "ai_recommends_approval": true
}
Placement for Ahmed Youssef (SPR-1001) was APPROVED by Nour (Career Advisor). AI recommendation was: approve. Notes: Confirmed.
